In [2]:
from pprint import pprint
from io_utils import load_document_annotation

json_path = r"C:\ann.JSOn"
doc = load_document_annotation(json_path)

print("doc type:", type(doc).__name__)
print("num_pages:", doc.num_pages)
print("tables:", len(doc.tables))
print("pages:", len(doc.pages))

assert doc.num_pages == len(doc.pages)
assert isinstance(doc.tables, list)
assert isinstance(doc.pages, list)
assert len(doc.pages) > 0

t0 = doc.tables[0]
print("\nfirst table:")
print(t0)

required_table_attrs = ["table_id", "page_index", "class_id", "rotated_90", "bbox"]
for attr in required_table_attrs:
    assert hasattr(t0, attr), attr

page0 = doc.pages[0]
print("\npage0 type:", type(page0).__name__)

# если page0 — dict
if isinstance(page0, dict):
    assert "ocr_rows" in page0
    ocr_rows = page0["ocr_rows"]
else:
    # если page0 тоже dataclass
    assert hasattr(page0, "ocr_rows")
    ocr_rows = page0.ocr_rows

print("ocr_rows:", len(ocr_rows))
assert len(ocr_rows) > 0

row0 = ocr_rows[0]
print("\nfirst ocr row:")
print(row0)

required_row_attrs = ["bbox", "text"]
for attr in required_row_attrs:
    assert hasattr(row0, attr), attr

assert isinstance(row0.bbox, list)
assert len(row0.bbox) == 4
assert isinstance(row0.text, str)

print("\nJSON loading test: OK")

doc type: DocumentAnnotation
num_pages: 49
tables: 44
pages: 49

first table:
PhysicalTable(table_id=1, page_index=5, class_id=1, rotated_90=False, bbox=[0.144204, 0.174, 0.876532, 0.617333])

page0 type: dict
ocr_rows: 6

first ocr row:
OcrRow(row_idx=0, bbox=[0.516494, 0.328, 0.722903, 0.342667], text='независимогоаудитора', text_conf=0.9840046763420105, table_id=None, table_id_prelabel=None)

JSON loading test: OK


In [9]:
import numpy as np

In [5]:
from io_utils import load_document_annotation, load_row_codes_txt

import importlib
import preprocessing as prep
importlib.reload(prep)


from features import (
#normalize_bbox_to_table,
    build_slot_numeric_features,
)
from config import NUM_BINS, NUM_SLOTS, SLOT_FEATURES

json_path = r"C:\\example_labels.json"
row_codes_path = "resources/row_codes.txt"

doc = load_document_annotation(json_path)
row_codes = load_row_codes_txt(row_codes_path)

table = next(t for t in doc.tables if not t.rotated_90)
rows = prep.extract_table_rows(doc, table)

print("table:", table)
print("rows in table:", len(rows))
assert len(rows) > 0, "У таблицы не нашлось OCR-строк"

r0 = rows[0]
print("\nfirst table row:", r0)

#norm_bbox = normalize_bbox_to_table(r0.bbox, table.bbox)
#print("norm_bbox:", norm_bbox)
#assert isinstance(norm_bbox, list)
#assert len(norm_bbox) == 4

bin_idx = prep.assign_row_to_page_bin(r0.bbox, NUM_BINS)
print("bin_idx:", bin_idx)
assert 0 <= bin_idx < NUM_BINS

feats = build_slot_numeric_features(r0.text, r0.bbox, row_codes)
print("\nSLOT_FEATURES:")
print(SLOT_FEATURES)
print("feature values:")
print(feats)

assert len(feats) == len(SLOT_FEATURES), (len(feats), len(SLOT_FEATURES))

sample = prep.build_table_sample(
    doc=doc,
    table=table,
    row_codes=row_codes,
    num_bins=NUM_BINS,
    num_slots=NUM_SLOTS,
)

assert sample is not None, "build_table_sample вернул None"
print("\nsample_id:", sample.sample_id)
print("class_id:", sample.class_id)
print("num_steps:", len(sample.steps))
assert len(sample.steps) > 0

for step0 in sample.steps:
    #step0 = sample.steps[0]
    print("\nstep0:")
    print("seq_pos:", step0.seq_pos)
    print("seq_len:", step0.seq_len)
    print("empty_bins_from_prev:", step0.empty_bins_from_prev)
    print("num_slots:", len(step0.slots))

    assert len(step0.slots) == NUM_SLOTS
    print("--------------------------------------")

    for i, slot in enumerate(step0.slots):
        print(f"slot {i}: text={repr(slot.text)}")
        print(f"slot {i}: n_features={len(slot.numeric_features)}")
        assert len(slot.numeric_features) == len(SLOT_FEATURES)

print("\nFeature pipeline test: OK")

table: PhysicalTable(table_id=1, page_index=5, class_id=1, rotated_90=False, bbox=[0.144204, 0.174, 0.876532, 0.617333])
rows in table: 68

first table row: OcrRow(row_idx=6, bbox=[0.809614, 0.596, 0.860509, 0.611333], text='35:о', text_conf=0.7189288139343262, table_id=None, table_id_prelabel=1)
bin_idx: 38

SLOT_FEATURES:
['x_left', 'x_right', 'width', 'y_center', 'row_height', 'text_len', 'num_words', 'num_digits_text', 'has_parentheses', 'has_slash', 'is_upper_like', 'is_total_like', 'has_digits', 'looks_like_row_code', 'looks_like_date']
feature values:
[0.809614, 0.860509, 0.050895000000000024, 0.6036665, 0.015333000000000041, 4.0, 1.0, 2.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]

sample_id: C:\example_labels.json::page5::table1
class_id: 1
num_steps: 20

step0:
seq_pos: 0
seq_len: 20
empty_bins_from_prev: 0
num_slots: 5
--------------------------------------
slot 0: text='декабря'
slot 0: n_features=15
slot 1: text='3а 12 месяцев, закончившихся'
slot 1: n_features=15
slot 2: text='1

In [ ]:
len(SLOT_FEATURES) == len(sample.steps[0].slots[0].numeric_features)


NameError: name 'GLOBAL_BIN_FEATURES' is not defined

In [8]:
len(GLOBAL_BIN_FEATURES) == 3

NameError: name 'GLOBAL_BIN_FEATURES' is not defined

In [4]:
import pandas as pd

rows = []

for step in sample.steps:
    row = {
        "seq_pos": step.seq_pos,
        "seq_len": step.seq_len,
        "empty_bins_from_prev": step.empty_bins_from_prev,
        "bin_index": step.bin_index,
    }

    assert len(step.slots) == NUM_SLOTS

    for slot_idx, slot in enumerate(step.slots):
        row[f"slot{slot_idx}_text"] = slot.text
        row[f"slot{slot_idx}_bbox"] = slot.bbox

        assert len(slot.numeric_features) == len(SLOT_FEATURES)

        for feat_name, feat_value in zip(SLOT_FEATURES, slot.numeric_features):
            row[f"slot{slot_idx}_{feat_name}"] = feat_value

    rows.append(row)

df_features = pd.DataFrame(rows)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(df_features)

print("\nFeature pipeline test: OK")

NameError: name 'sample' is not defined

In [1]:
from config import NUM_BINS, NUM_SLOTS, SPLIT_XLS_PATH, ROW_CODES_PATH

from config import SLOT_FEATURES

from preprocessing import build_dataset_splits

train_samples, val_samples, test_samples = build_dataset_splits(
    split_xls_path=SPLIT_XLS_PATH,
    row_codes_path=ROW_CODES_PATH,
    num_bins=NUM_BINS,
    num_slots=NUM_SLOTS,
)

print("=== Dataset sizes ===")
print("train:", len(train_samples))
print("val  :", len(val_samples))
print("test :", len(test_samples))

# быстрые sanity-check-и
def check_samples(name, samples):
    print(f"\nChecking {name} ({len(samples)} samples)")
    for i, sample in enumerate(samples[:10]):  # первые 10
        assert sample.steps, "sample without steps"
        for step in sample.steps:
            assert len(step.slots) == NUM_SLOTS
            for slot in step.slots:
                assert len(slot.numeric_features) == len(SLOT_FEATURES)
    print(f"{name} ok")

check_samples("train", train_samples)
check_samples("val", val_samples)
check_samples("test", test_samples)

print("\nFull dataset load + feature generation: OK")

=== Dataset sizes ===
train: 3117
val  : 771
test : 811

Checking train (3117 samples)
train ok

Checking val (771 samples)
val ok

Checking test (811 samples)
test ok

Full dataset load + feature generation: OK


In [2]:
import numpy as np

def summarize_samples(name, samples):
    seq_lens = [len(s.steps) for s in samples]
    num_steps = len(seq_lens)
    all_bin_idx = [step.bin_index for s in samples for step in s.steps]
    empty_slots = 0
    total_slots = 0

    for s in samples:
        for step in s.steps:
            for slot in step.slots:
                total_slots += 1
                if not slot.text:
                    empty_slots += 1

    print(f"\n{name}:")
    print("  #samples:", len(samples))
    print("  seq_len:  min", np.min(seq_lens), "max", np.max(seq_lens),
          "mean", float(np.mean(seq_lens)))
    print("  bin_index: min", min(all_bin_idx), "max", max(all_bin_idx))
    print("  empty slots: {:.1%}".format(empty_slots / max(1, total_slots)))

summarize_samples("train", train_samples)
summarize_samples("val", val_samples)


train:
  #samples: 3117
  seq_len:  min 1 max 59 mean 14.847930702598653
  bin_index: min 0 max 63
  empty slots: 41.7%

val:
  #samples: 771
  seq_len:  min 1 max 58 mean 15.59662775616083
  bin_index: min 2 max 61
  empty slots: 41.9%


In [12]:
from collections import Counter

label_counts = Counter(s.class_id for s in train_samples)
print("Train class counts:")
for cls_id, cnt in sorted(label_counts.items()):
    print(f"  class {cls_id}: {cnt}")

Train class counts:
  class 0: 191
  class 1: 166
  class 2: 127
  class 3: 109
  class 4: 2524


In [13]:
pip install pipreqs

  Using cached nbconvert-7.17.1-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/798.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/798.3 kB ? eta -:--:--
   ---------------------------------------- 798.3/798.3 kB 3.8 MB/s  0:00:00

   ----------------------------------------  0/11 [pickleshare]
   --- ------------------------------------  1/11 [backcall]
   --- ------------------------------------  1/11 [backcall]
   --- ------------------------------------  1/11 [backcall]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------------  2/11 [tinycss2]
   ------- --------------------------

In [1]:
pip install pipreqs

Note: you may need to restart the kernel to use updated packages.
